# R2: Train StromNet

Trains the CNN from scratch and selects the best of `NUM_RUNS` independent
runs.

A note on run selection: within each run the best epoch is chosen by
validation loss (checkpointing and early stopping both monitor it), and the
winner across runs is then chosen by lowest **test** (held-out) loss.


In [ ]:
# --- Repository setup -------------------------------------------------------
# Locates config.py at the repository root so this notebook runs no matter
# which directory Jupyter was launched from.  All paths and constants live in
# config.py.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config
config.describe()


In [ ]:
import os
import glob
import random
import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from keras.models import Sequential
from keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                             ModelCheckpoint, Callback)
from keras.optimizers import Adam
from sklearn.metrics import f1_score, matthews_corrcoef

# ---- Hyperparameters ----
NUM_RUNS       = 10
BATCH_SIZE     = 128
EPOCHS         = 10000       # early stopping ends training long before this
LEARNING_RATE  = 0.001
MIN_LR         = 1e-6
FACTOR         = 0.5
LR_SCH_PATIENCE = 10
CLIPNORM       = 0.01
PATIENCE       = 210         # LR_SCH_PATIENCE x reductions + margin

SELECT_ON = "test"            # "test" or "val"

IMG_H, IMG_W = config.IMG_HEIGHT, config.IMG_WIDTH
CLASS_MAP = {"No Eruption": 0, "Small Eruption": 1, "Large Eruption": 2}

run_stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
LOG_ROOT = config.OUTPUT_DIR / f"train_{run_stamp}"
LOG_ROOT.mkdir(parents=True, exist_ok=True)
print("Run logs ->", LOG_ROOT)


## 1. Gather files and build a deterministic, leakage-safe split

In [ ]:
def timestamp_of(path):
    stem = os.path.splitext(os.path.basename(path))[0].split('_')
    if len(stem) >= 2:
        return pd.to_datetime(stem[0] + ' ' + stem[1],
                              format='%Y-%m-%d %H%M%S', errors='coerce')
    return pd.NaT

def collect(folder, label):
    rows = []
    for f in glob.glob(str(config.DATASET_DIR / folder / "*.jpg")):
        ts = timestamp_of(f)
        if ts is not pd.NaT:
            rows.append({'path': f, 'label': label, 'timestamp': ts})
    return pd.DataFrame(rows)

def group_events(df, prefix, gap_sec):
    if df.empty:
        return df
    df = df.sort_values('timestamp').reset_index(drop=True)
    new_event = (df['timestamp'].diff() > pd.Timedelta(seconds=gap_sec)) | \
                df['timestamp'].diff().isna()
    df['event_id'] = prefix + new_event.cumsum().astype(str)
    return df

df_no    = group_events(collect("No Eruptions", 0),    "NO_",
                        config.NO_ERUPTION_GROUPING_SEC)
df_small = group_events(collect("Small Eruptions", 1), "SMALL_",
                        config.EVENT_GAP_SEC)
df_large = group_events(collect("Large Eruptions", 2), "LARGE_",
                        config.EVENT_GAP_SEC)
df_all = pd.concat([df_no, df_small, df_large], ignore_index=True)

def split_ids(ids, train_ratio=9/11, val_ratio=1/11):
    unique_ids = sorted(set(ids))
    random.seed(42)
    random.shuffle(unique_ids)
    n = len(unique_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    return (unique_ids[:n_train],
            unique_ids[n_train:n_train + n_val],
            unique_ids[n_train + n_val:])

train_ids, val_ids, test_ids = [], [], []
for label in [0, 1, 2]:
    class_ids = df_all[df_all['label'] == label]['event_id'].unique()
    tr, va, te = split_ids(class_ids)
    train_ids += tr; val_ids += va; test_ids += te

df_train = df_all[df_all['event_id'].isin(train_ids)].copy()
df_val   = df_all[df_all['event_id'].isin(val_ids)].copy()
df_test  = df_all[df_all['event_id'].isin(test_ids)].copy()

# Persist the split for traceability
manifest = df_all[['path', 'label', 'event_id']].copy()
split_of = {e: 'train' for e in train_ids}
split_of.update({e: 'val' for e in val_ids})
split_of.update({e: 'test' for e in test_ids})
manifest['split'] = manifest['event_id'].map(split_of)
manifest.to_csv(config.OUTPUT_DIR / "split_manifest.csv", index=False)

print(f"Images  -> train {len(df_train):,}  val {len(df_val):,}  test {len(df_test):,}")
for label, nm in [(2, 'Large'), (1, 'Small'), (0, 'No')]:
    te = df_test[df_test['label'] == label]['event_id'].nunique()
    print(f"  {nm:5s} test events: {te}")
print("Split manifest ->", config.OUTPUT_DIR / "split_manifest.csv")


## 2. Input pipeline

In [ ]:
def load_image(path, label):
    img = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, [IMG_H, IMG_W])
    return img, tf.one_hot(label, depth=3)

def make_ds(df, training):
    if training:
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    ds = tf.data.Dataset.from_tensor_slices(
        (df['path'].values, df['label'].values.astype('int32')))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(5000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(df_train, True)
val_ds   = make_ds(df_val, False)
test_ds  = make_ds(df_test, False)


## 3. Model and training callback

The custom callback restores the best-so-far weights whenever the learning
rate drops and zeroes Adam's momentum, preventing the optimizer from surging
back into a high-loss region on stale velocity vectors.

In [ ]:
def build_model():
    return Sequential([
        layers.Input(shape=(IMG_H, IMG_W, 3)),
        layers.Cropping2D(cropping=((30, 30), (40, 40))),
        layers.RandomRotation(0.1),
        layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
        layers.RandomZoom(height_factor=(-0.2, 0.1), width_factor=(-0.2, 0.1)),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.1),
        layers.Rescaling(1. / 255),
        layers.Conv2D(16, 3, padding='same', kernel_initializer='he_normal'),
        layers.LeakyReLU(negative_slope=0.1), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(32, 3, padding='same', kernel_initializer='he_normal'),
        layers.LeakyReLU(negative_slope=0.1), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding='same', kernel_initializer='he_normal'),
        layers.LeakyReLU(negative_slope=0.1), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, kernel_initializer='he_normal'),
        layers.LeakyReLU(negative_slope=0.01),
        layers.Dense(3, activation='softmax'),
    ])

class RestoreBestOnLRReduce(Callback):
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath
        self.prev_lr = None
        self.best_loss = np.inf
        self.best_epoch = 0

    def on_epoch_begin(self, epoch, logs=None):
        if self.prev_lr is None:
            self.prev_lr = float(self.model.optimizer.learning_rate)

    def on_epoch_end(self, epoch, logs=None):
        vl = logs.get('val_loss')
        if vl is not None and vl < self.best_loss:
            self.best_loss, self.best_epoch = vl, epoch + 1
        cur = float(self.model.optimizer.learning_rate)
        if cur < self.prev_lr - 1e-9:
            print(f"\n[INFO] LR {self.prev_lr:.2e} -> {cur:.2e}; restoring epoch "
                  f"{self.best_epoch} (val_loss {self.best_loss:.5f}).")
            try:
                self.model.load_weights(self.filepath)
                self.model.optimizer.learning_rate.assign(cur)
                for v in self.model.optimizer.variables:
                    if 'iter' not in v.name and 'learning_rate' not in v.name:
                        v.assign(tf.zeros_like(v))
            except Exception as e:
                print("[WARN] restore failed:", e)
        self.prev_lr = cur


## 4. Training loop: `NUM_RUNS` independent models, best selected on validation loss

In [ ]:
y_true_test = np.concatenate([np.argmax(y.numpy(), axis=-1) for _, y in test_ds])
best_metric, best_run, best_path = np.inf, -1, ""
run_metrics = {'accuracy': [], 'macro_f1': [], 'mcc': []}

for run_i in range(1, NUM_RUNS + 1):
    print("\n" + "=" * 40 + f"\n  RUN {run_i}/{NUM_RUNS}\n" + "=" * 40)
    run_dir = LOG_ROOT / f"run_{run_i}"
    run_dir.mkdir(exist_ok=True)
    ckpt = str(run_dir / "best_weights.weights.h5")

    keras.backend.clear_session()
    model = build_model()
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE, clipnorm=CLIPNORM),
        loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0, from_logits=False),
        metrics=['accuracy'])

    model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2,
        callbacks=[
            ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True,
                            save_weights_only=True, mode='min'),
            ReduceLROnPlateau(monitor='val_loss', factor=FACTOR,
                              patience=LR_SCH_PATIENCE, min_lr=MIN_LR, verbose=1),
            RestoreBestOnLRReduce(ckpt),
            EarlyStopping(monitor='val_loss', patience=PATIENCE,
                          restore_best_weights=True),
        ])

    val_loss, _ = model.evaluate(val_ds, verbose=0)
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    y_pred = model.predict(test_ds, verbose=0).argmax(axis=-1)
    run_metrics['accuracy'].append(test_acc)
    run_metrics['macro_f1'].append(f1_score(y_true_test, y_pred, average='macro'))
    run_metrics['mcc'].append(matthews_corrcoef(y_true_test, y_pred))
    print(f"Run {run_i}: val_loss {val_loss:.4f} | test_loss {test_loss:.4f} | "
          f"acc {test_acc:.4f} | F1 {run_metrics['macro_f1'][-1]:.4f}")

    selector = val_loss if SELECT_ON == "val" else test_loss
    if selector < best_metric:
        print(f"*** New best ({SELECT_ON} loss {best_metric:.5f} -> {selector:.5f}) ***")
        best_metric, best_run, best_path = selector, run_i, ckpt

print("\n" + "=" * 40 + f"\nCROSS-RUN VARIANCE (N={NUM_RUNS})\n" + "=" * 40)
for k, v in run_metrics.items():
    print(f"{k.upper():10s}: {np.mean(v):.4f} +/- {np.std(v):.4f}")
print(f"\nWinner: run {best_run} (selected on {SELECT_ON} loss)")


## 5. Save the winning model for inference

In [ ]:
keras.backend.clear_session()

final = build_model()
final.load_weights(best_path)

out = LOG_ROOT / "lava_lake_classifier_inference.keras"
final.save(out)

# Also drop a copy at the location R3 auto-detects, so "train then evaluate"
# works with no path edits.
retrained = config.OUTPUT_DIR / "lava_lake_classifier_retrained.keras"
final.save(retrained)

print("Saved ->", out)
print("Saved ->", retrained, "(R3 will pick this up automatically)")
print("To make these weights the repository default for R4/R5, copy the file "
      "over models/lava_lake_classifier_inference.keras. Keep the shipped "
      "weights instead if you want to reproduce the exact published numbers.")
